# SFMTL Client

> The core abstraction for `SFMTL` client.

In [ ]:
#| default_exp clients.sfmtl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from fedai.clients.base_client import BaseClient
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from fedai.clients import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np



In [ ]:
#| export
class SFMTLClient(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ): 
        

        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        self.h_c = self.h_c if hasattr(self, 'h_c') else None
        self.anchorloss = AnchorLoss(random_seed= self.cfg.random_seed, 
                                     num_classes= self.cfg.data.num_classes,
                                     hidden_dim= self.cfg.model.hidden_dim,
                                     t= self.t, 
                                     h_c= self.h_c).to(self.device)

        self.label_set = list(set(np.array([batch[self.label_key] for batch in self.train_loader.dataset])))

### Training

In [ ]:
#| export
@patch
def fit(self: SFMTLClient):
    
    self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        epoch_mean_anchor = torch.zeros(self.cfg.data.num_classes, self.cfg.model.hidden_dim).to(self.device)
        with torch.no_grad():
            epoch_mean_anchor.copy_(self.anchorloss.anchor)

        for batch_idx, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)

            batch_mean_anchor = torch.zeros(self.cfg.data.num_classes, self.cfg.model.hidden_dim).to(self.device)
            self.optimizer.zero_grad()

            X, y = batch[self.data_key], batch[self.label_key]
            y_copied = deepcopy(y)
            labels = y.type(torch.LongTensor).to(self.device)
            ys = labels.float()
            
            h = self.model.backbone(X)
            outputs = self.model.head(h)    
            
            loss_anchor = self.anchorloss(h, ys, Lambda = self.cfg.lambda_anchor)
            loss = self.criterion(outputs, y_copied) + loss_anchor
            
            for i in set(labels.tolist()):
                batch_mean_anchor[i] += torch.mean(h[labels==i], dim= 0)
        
            loss.backward()
            self.optimizer.step()

            for i in self.label_set:
                #compute batch mean anchor according to batch label
                batch_mean_anchor[i] = batch_mean_anchor[i]/(batch_idx+1)
                #compute epoch mean anchor according to batch mean anchor
                lambda_momentum = self.cfg.momentum_anchor #pow(2, -(epoch+1))
                # epoch_mean_anchor[i] = lambda_momentum * epoch_mean_anchor[i] + (1-lambda_momentum)*batch_mean_anchor[i]
                epoch_mean_anchor[i].mul_(lambda_momentum).add_((1 - lambda_momentum) * batch_mean_anchor[i])        

        with torch.no_grad():
            self.anchorloss.anchor.copy_(epoch_mean_anchor)



### Communication

The communication in SFMTL is a matter of sending (saving to the disk) two things, the classification head and a randomly picked data represntation.

In [ ]:
#| export
@patch
def save_state(self: SFMTLClient, save_to_disk= False):
    """
    Save client's state including model, optimizer, anchor, and label set to a dictionary. Optionally, save the state to disk.
    """
    
    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()
    state_dict['h'] = self.anchorloss.anchor.detach().clone() # (num_classes, hidden_size)
    state_dict['label_set'] = self.label_set
    
    if save_to_disk:
        state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)
    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()